In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS, savefig

Definition of Model Regions for Austria

In [ ]:
import geopandas as gpd

gadm_path = RAW / "gadm" / "gadm_410-levels-ADM_1-AUT.gpkg"

gadm = gpd.read_file(gadm_path)

# Show basic information
print("Columns:", gadm.columns.tolist())
print("CRS:", gadm.crs)
print("Number of regions:", len(gadm))
print("Country codes:", gadm["GID_0"].unique())
print("Country names:", gadm["COUNTRY"].unique())
print(gadm[["GID_1", "NAME_1"]])

# Save clean copy
gadm.to_file(PROCESSED / "gadm_aut_level1.geojson", driver="GeoJSON")

EEZ : Austria is a landlocked country; therefore, no Marine Regions or EEZ data are required.

Define Centroids of Each Region

In [ ]:
# Project to a metric CRS, since .centroid works with projected coordinates.
gadm_projected = gadm.to_crs("EPSG:3035")

# Calculate centroids and convert back to WGS84
centroids = gadm_projected.geometry.centroid.to_crs("EPSG:4326")

# Create a new dataframe containing only region names and centroids
region_centroids = gpd.GeoDataFrame(
    {
        "GID": gadm["GID_1"],
        "region": gadm["NAME_1"],
        "geometry": centroids
    },
    crs="EPSG:4326"
)

# Longitude and latitude
region_centroids["x"] = region_centroids.geometry.x
region_centroids["y"] = region_centroids.geometry.y

# Save centroids
region_centroids.to_file(
    PROCESSED / "austria_region_centroids.geojson",
    driver="GeoJSON"
)

In [ ]:
import matplotlib.pyplot as plt

#Visual for centroids
ax = gadm.plot(figsize=(10, 6),edgecolor="black",facecolor="white",linewidth=0.8)

plt.scatter(region_centroids["x"],region_centroids["y"],marker="x",s=80,color="crimson",zorder=3)

for _, row in region_centroids.iterrows():
    ax.annotate(
        row["region"],
        xy=(row["x"], row["y"]),
        xytext=(row["x"] + 0.05, row["y"] + 0.05),
        fontsize=8,
        weight="bold",
        color="black",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.7),
        zorder=4,
    )

plt.title("Centroids of Austrian Regions")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

savefig(ax.figure, "regions", "centroids_map.png")
plt.show()